# **Lab 5: Custom Data Sources**

Lecturer: `Sirasit Lochanachit`


Course: `06026213 Big Data Systems`

Term: `02/2025`

Lab materials prepared by `Sasithorn (TA) - 2025`

PySpark Custom Data Sources คือ การใช้ PySpark DataSource API ที่ช่วยให้สามารถอ่านข้อมูลจากแหล่งข้อมูลแบบ custom ได้เอง และยังสามารถเขียนไปยังปลายทางได้แบบ custom เองใน Apache Spark ได้ โดยใช้ Python

ซึ่งในไฟล์นี้ จะมีด้วยกัน 2 ตัวอย่างดังนี้
- Batch Query
- GitHub DataSource

Note: PySpark DataSource คือ base class ที่มี method ในการสร้าง data readers and writers

### Implement the data source subclass

ในการที่จะทำให้ data source สามารถ read หรือ write ได้จะต้องมีการ implement subclass ที่มี propery หรือ method ดังต่อไปนี้

| Property / Method | Description |
|------------------------------------|---------------------------------------------------------------------------|
| name [**Required**] |  เป็นการระบุชื่อของ Data Source |
| schema [**Required**] |  เป็นการระบุโครงสร้างข้อมูล (Schema) ของ Data Source ที่จะอ่านหรือเขียน |
| reader() | ต้องคืนค่า `DataSourceReader` เพื่อให้ Data Source อ่านข้อมูลแบบ Batch |
| writer() | ต้องคืนค่า `DataSourceWriter` เพื่อให้ Data Sink เขียนข้อมูลแบบ Batch |
| streamReader() / simpleStreamReader() | ต้องคืนค่า `DataSourceStreamReader` เพื่อให้ Data Stream อ่านข้อมูลแบบ Streaming |
| streamWriter() | ต้องคืนค่า `DataSourceStreamWriter` เพื่อให้ Data Stream เขียนข้อมูลแบบ Streaming |


# Example 1: Create a PySpark DataSource for batch query

ตัวอย่าง: ใช้ faker มาช่วยในการจำลองข้อมูลตัวอย่าง

[Faker](https://faker.readthedocs.io/en/master/)

In [0]:
%pip install faker

หลังจาก install จะมี Note บอกให้ restart kernel โดยใช้ `%restart_python` หรือ `dbutils.library.restartPython()`

(ในที่นี้เลยเลือกใช้ `%restart_python`) แล้วกดรันใหม่อีกรอบ จะขึ้นแจ้งเตือนเหมือนเดิม แต่สามารถใช้งาน faker lib ได้แล้ว

In [0]:
%restart_python

## Step 1: Define the example DataSource

สร้าง Subclass จาก DataSource หลักสำหรับแหล่งข้อมูลขึ้นมาก่อน โดยจะมีการกำหนด 3 ส่วนใน Subclass นี้คือ
- name : ชื่อ DataSource Class ที่จะใช้กับ spark
- schema : โครงสร้างข้อมูลที่จะอ่าน ว่ามี column อะไรบ้าง, data type เป็นยังไง 
- reader : อ่านข้อมูลจาก Data Source โดยใช้ batch query

In [0]:
from pyspark.sql.datasource import DataSource, DataSourceReader
from pyspark.sql.types import StructType

class FakeDataSource(DataSource):
    """
    An example data source for batch query using the `faker` library.
    """

    @classmethod
    def name(cls):
        return "fake"

    def schema(self):
        return "name string, date string, zipcode string, state string"

    def reader(self, schema: StructType):
        return FakeDataSourceReader(schema, self.options)

## Step 2: Implement the reader for a batch query

กำหนด reader logic เพื่อสร้างข้อมูลตัวอย่าง โดยในตัวอย่างนี้ใช้ faker library ในการเติมแต่ละ field ข้อมูลใน schema
- จำนวน rows โดย default = 3


In [0]:
class FakeDataSourceReader(DataSourceReader):

    def __init__(self, schema, options):
        self.schema: StructType = schema
        self.options = options

    def read(self, partition):
        # Library imports must be within the method.
        from faker import Faker
        fake = Faker()

        # Every value in this `self.options` dictionary is a string.
        # numRows: specify number of rows to generate. Default value is 3.
        num_rows = int(self.options.get("numRows", 3))
        for _ in range(num_rows):
            row = []
            for field in self.schema.fields:
                value = getattr(fake, field.name)()
                row.append(value)
            yield tuple(row)

จากโค้ด ใน method `read()` มีจุดให้สังเกตเพิ่มเติม 2 ส่วน
1. ข้อมูลตัวอย่างถูกสร้างจาก lib faker ที่เราติดตั้งเมื่อก่อนหน้านี้นั่นเอง
2. จะเห็นว่า method นี้ใช้คำสั่ง `yield` ในการส่งข้อมูลกลับออกมา ซึ่งคำสั่งนี้มันจะทำการส่งค่าออกมาทีละค่า แล้ว pause ไว้ พอถูกเรียกครั้งต่อไป มันก็จะทำงานต่อจากจุดเดิม ซึ่งจะมีผลดีเมื่อเจอข้อมูลที่มันใหญ่มากๆ

## Step 3: Register and use the example data source

- การที่จะใช้งาน Data source ได้นั้นจะต้องทำการ register ก่อน
- ตัวอย่าง Code ด้านล่างเรียกทำการ 
  1. register data source subclass 
  2. จากนั้นเรียกใช้งาน data source "fake" 
  3. และทำการ load ข้อมูลจาก data source โดยใช้ค่า default
      - return num_rows = 3
      - return `name string, date string, zipcode string, state string`

In [0]:
spark.dataSource.register(FakeDataSource)
spark.read.format("fake").load().show()

- ถ้าต้องการเพียงแค่ 2 คอลัมน์ก็ทำได้ ให้ระบุชื่อ column/field ที่ต้องการใน schema ได้เลย
- ในตัวอย่างนี้มีการระบุ schema เพิ่มเติม (custom) จาก default schema ใน class คือ `company` (override ค่า default)

In [0]:
spark.read.format("fake").schema("name string, company string").load().show()

- สามารถระบุจำนวนแถวที่ต้องการได้ เช่น 5 แถว (override ค่า default)

In [0]:
spark.read.format("fake").option("numRows", 5).load().show()

---

# Example 2: Create a PySpark GitHub DataSource

%md
Example นี้จะสาธิตการดึงข้อมูล Pulls จาก GitHub repo - https://github.com/apache/spark มาแสดงเป็นตารางใน PySpark

## Step 1: Define the GitHub DataSource

ขั้นแรก : Define Class หลักสำหรับแหล่งข้อมูลขึ้นมา โดยจะมี 3 ส่วนใน Class คือ
- name : ชื่อ DataSource Class ที่จะใช้สำหรับใช้กับ spark
- schema : โครงสร้างข้อมูลที่จะอ่าน ว่ามี column อะไรบ้าง, data type เป็นยังไง 
- reader : ตัวอ่านข้อมูลจาก DataSource (implementation)

ซึ่งส่วนนี้จะเหมือนกันกับตัวอย่างแรก

In [0]:
import json
import requests

from pyspark.sql import Row
from pyspark.sql.datasource import DataSource, DataSourceReader
from pyspark.sql.types import VariantVal

class GithubVariantDataSource(DataSource):
    @classmethod
    def name(self):
        return "githubVariant"
    def schema(self):
        return "id int, title string, user string, created_at string, updated_at string"
    def reader(self, schema):
        return GithubVariantPullRequestReader(self.options)


## Step 2: Implement the reader to retrieve pull requests

ส่วนนี้คือส่วนของการสร้างตัวอ่านข้อมูล (Reader) แต่จะมีความซับซ้อนกว่าตัวอย่างแรก เนื่องจากเป็นการดึงข้อมูลจาก GitHub

มี 2 ส่วน คือ
1. `__init__` ส่วนนี้จะเป็นส่วนที่รับ path ของ repo, token(ถ้ามี) หาก repo ว่าง จะไม่สามารถทำงานต่อได้
2. `read` ส่วนนี้จะมีการทำงานย่อยลงไปอีก กล่าวคือ
> - เตรียม HTTP Header
> - Request & Response
> - แปลงเป็น row ส่งกลับ (yield rows)

In [0]:
class GithubVariantPullRequestReader(DataSourceReader):
    def __init__(self, options):
        self.token = options.get("token")
        self.repo = options.get("path")
        if self.repo is None:
            raise Exception(f"Must specify a repo in `.load()` method.")
        # ทุก value ใน self.options dictionary เป็น string
        self.num_rows = int(options.get("numRows", 10))

    def read(self, partition):
        header = {
            "Accept": "application/vnd.github+json", # ต้องการข้อมูลแบบ JSON
        }
        if self.token is not None:                              # ถ้ามี token ให้แนบไปด้วย
            header["Authorization"] = f"Bearer {self.token}"
        url = f"https://api.github.com/repos/{self.repo}/pulls" # เตรียม address ปลายทาง
        response = requests.get(url, headers=header)            # สร้างตัวแปร response เพื่อไปรับผลจากการ request
        response.raise_for_status()                             # ตรวจสอบ status
        prs = response.json()                                   # แปลงเป็น json
        for pr in prs[:self.num_rows]:                          # loop ส่งข้อมูลกลับออกมาเป็น row
            yield Row(
                id = pr.get("number"),
                title = pr.get("title"),
                user = pr.get("user"),
                created_at = pr.get("created_at"),
                updated_at = pr.get("updated_at")
            )

## Step 3: Register and use the data source

เหมือนเดิมจากที่เราเคยทำในตัวอย่างแรก เมื่อเราสร้าง custom data source ขึ้นมา ต้องมีการ `register` ให้ spark รู้จักก่อนจึงจะใช้งานได้

In [0]:
spark.dataSource.register(GithubVariantDataSource)
spark.read.format("githubVariant").option("numRows", 3).load("apache/spark").display()

---

# Take-home Exercise

1. สร้าง Custom Data Source ที่ดึงข้อมูลผ่าน Public/Private API

- https://www.databricks.com/blog/announcing-general-availability-python-data-source-api

---

References
- [Data Source Base Class](https://github.com/apache/spark/blob/0d7c07047a628bd42eb53eb49935f5e3f81ea1a1/python/pyspark/sql/datasource.py)
- Built-in DataFrameReader for various formats
    - https://api-docs.databricks.com/python/pyspark/latest/pyspark.sql/io.html
    - https://api-docs.databricks.com/python/pyspark/latest/pyspark.pandas/io.html

---